# Chapter 4 (part 1) — Simple and Multiple Linear Regression

Covers the first part of Ch 4 of *Practical Statistics for Data Scientists*: Simple Linear Regression (equation, fitted values, residuals) → Multiple Linear Regression → Assessing the Model (RMSE, R², adjusted R²) → Model Selection and Stepwise Regression → Weighted Regression, all on Ames Housing with `Sale_Price` as the target.

The thread running through this notebook: **a regression is a machine for answering "how much does y move when x moves a bit, with everything else held where it is?"** — and almost every pitfall (coefficients that change when you add a variable, R² that flatters you, stepwise results that don't replicate) comes from forgetting what "everything else" means or what the model was allowed to see.

Ask for a hint if you get stuck; per the repo's review protocol, the first couple of asks per question get a nudge, not the answer. The exam at the bottom is the part for checking whether the intuition stuck.

Primary tools: `statsmodels.formula.api.ols` for fitting and reading summaries, pandas / numpy for the by-hand checks, `sklearn.model_selection.train_test_split` for hold-out splits.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

(2930, 81)

## 1. Simple Linear Regression

Simple linear regression draws the one straight line `Sale_Price ≈ b0 + b1 * Gr_Liv_Area` that makes the squared vertical misses as small as possible.

Numeric picture first. Take three houses: (1000 sqft, $100k), (1500 sqft, $140k), (2000 sqft, $200k). The average house is 1500 sqft and $146,667. The slope is "how do the dollar-gaps from average line up with the sqft-gaps from average?" — houses are -500, 0, +500 sqft from the mean and -46,667, -6,667, +53,333 dollars from the mean. Multiply paired gaps and add: (-500)(-46,667) + 0 + (500)(53,333) = 50,000,000; divide by the sum of squared sqft-gaps (250,000 + 0 + 250,000 = 500,000) and you get b1 = $100 per sqft. The intercept is whatever makes the line pass through the average point: 146,667 - 100(1500) = -$3,333. The *fitted value* for a house is the line's height at its sqft; the *residual* is actual minus fitted.

**Question 1a.** Fit `Sale_Price ~ Gr_Liv_Area` on the full data with `smf.ols`. Report the slope and intercept in plain dollars-per-square-foot terms. Does the intercept mean anything you'd say out loud? Then confirm, *without* using the fit, that the slope equals `corr(x, y) * std(y) / std(x)` and also `cov(x, y) / var(x)` — and say in a sentence why correlation and slope are different numbers.

*Tools for this question:*
- `smf.ols("Sale_Price ~ Gr_Liv_Area", data=df).fit()` — formula is `y ~ x1 + x2`, the intercept is added automatically. Read results with `.params` (coefficients), `.bse` (standard errors), `.summary()`.
- `s1.corr(s2)` Pearson r; `s.std()` / `s.var()` use n-1 by default; `s1.cov(s2)` covariance.

In [2]:
# part 1 fit the model
model = smf.ols("Sale_Price ~ Gr_Liv_Area",data = ames).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             Sale_Price   R-squared:                       0.500
Model:                            OLS   Adj. R-squared:                  0.499
Method:                 Least Squares   F-statistic:                     2923.
Date:                Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:05:14   Log-Likelihood:                -36218.
No. Observations:                2930   AIC:                         7.244e+04
Df Residuals:                    2928   BIC:                         7.245e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept    1.329e+04   3269.703      4.064      0.000    6878.485    1.97e+04
Gr_Liv_Area   111.6940      2.066     54.061      0.000     107.643     115.745
==============================================================================
Omnibus:                      491.939   Durbin-Watson:                   1.068
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             5135.212
Skew:                           0.468   Prob(JB):                         0.00
Kurtosis:                       9.418   Cond. No.                     4.96e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.96e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**Model Coefficient Results In Plain Language**
* Slope of Gr_Live_Area: Homes with one unit more in Gr_Liv_Area(so this may be every sq ft in living area) are on average $111.69 more expensive in terms of sales price.
* The intercept implies that a home with 0 sq ft of Gr_Liv_Area will still be valued at $13,290. Not something worth saying outloud.

In [3]:
slope = (ames["Gr_Liv_Area"].corr(ames["Sale_Price"])) * (np.std(ames["Sale_Price"])/np.std(ames["Gr_Liv_Area"]))
print(f"Slope using std and correlation: {slope}")

# confirm that slope matches up with its formula using var and cov
slope = (ames["Gr_Liv_Area"].cov(ames["Sale_Price"]))/np.var(ames["Gr_Liv_Area"], ddof=1)
print(f"Slope using cov and var: {slope}")

# print correlation between the 2 variables
print(f"Correlation: {ames["Gr_Liv_Area"].corr(ames["Sale_Price"])}")

Slope using std and correlation: 111.69400085780035
Slope using cov and var: 111.69400085780038
Correlation: 0.7067799209766278


* The slope computed by hand matches up with the slop computed using the package.
* Correlation differs from slope in the sense that it measures the relationship and strength of such relationship accounting for different units. Meanwhile, slope in this case takes the unit of dollars so show its relationship.

**Question 1b.** From the fitted model pull the residuals. Check numerically: (i) they sum to (essentially) zero, (ii) they're uncorrelated with `Gr_Liv_Area`. These aren't coincidences — in a sentence or two, explain why the least-squares line is *forced* to have both properties. Then: which three houses have the biggest positive residuals and which three the biggest negative? What's different about them in terms of columns you haven't put in the model (look at `Neighborhood`, `Overall_Qual`, `Sale_Condition`)?

*Tools for this question:*
- `res.resid` residuals and `res.fittedvalues` fitted values (both indexed like `df`); `np.corrcoef(a, b)[0, 1]` for the correlation.
- `df.assign(resid=res.resid).nlargest(3, "resid")` / `.nsmallest(3, "resid")` to pull the extreme rows.

In [4]:
# check the residuals and their sum
residual_sum = model.resid.sum()
print(f"residual sum: {round(residual_sum,2)}")

# check the residual correlation with dependent variable
print(f"Residual and Gr_Liv_Area correlation: {round(np.corrcoef(model.resid, ames["Gr_Liv_Area"])[0,1],2)}")

# 3 largest points 
ames.assign(resid = model.resid).nlargest(3, "resid")

residual sum: -0.0
Residual and Gr_Liv_Area correlation: 0.0


,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,Land_Slope,Neighborhood,Condition_1,Condition_2,Bldg_Type,House_Style,Overall_Qual,Overall_Cond,Year_Built,Year_Remod_Add,Roof_Style,Roof_Matl,Exterior_1st,Exterior_2nd,Mas_Vnr_Type,Mas_Vnr_Area,Exter_Qual,Exter_Cond,Foundation,Bsmt_Qual,Bsmt_Cond,Bsmt_Exposure,BsmtFin_Type_1,BsmtFin_SF_1,BsmtFin_Type_2,BsmtFin_SF_2,Bsmt_Unf_SF,Total_Bsmt_SF,Heating,Heating_QC,Central_Air,Electrical,First_Flr_SF,Second_Flr_SF,Low_Qual_Fin_SF,Gr_Liv_Area,Bsmt_Full_Bath,Bsmt_Half_Bath,Full_Bath,Half_Bath,Bedroom_AbvGr,Kitchen_AbvGr,Kitchen_Qual,TotRms_AbvGrd,Functional,Fireplaces,Fireplace_Qu,Garage_Type,Garage_Finish,Garage_Cars,Garage_Area,Garage_Qual,Garage_Cond,Paved_Drive,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,Three_season_porch,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Year_Sold,Sale_Type,Sale_Condition,Sale_Price,Longitude,Latitude,resid
44,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,100,12919,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Inside,Gtl,Northridge_Heights,Norm,Norm,OneFam,One_Story,Excellent,Average,2009,2010,Hip,CompShg,VinylSd,VinylSd,Stone,760,Excellent,Typical,PConc,Excellent,Typical,Gd,GLQ,3,Unf,0,142,2330,GasA,Excellent,Y,SBrkr,2364,0,0,2364,1,0,2,1,2,1,Excellent,11,Typ,2,Good,Attchd,Fin,3,820,Typical,Typical,Paved,0,67,0,0,0,0,No_Pool,No_Fence,NaN,0,3,2010,New,Partial,611657,-93.655051,42.059617,334322.747607
1063,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,106,12720,Pave,No_Alley_Access,Regular,HLS,AllPub,Inside,Mod,Northridge_Heights,Norm,Norm,OneFam,One_Story,Very_Excellent,Average,2003,2003,Hip,CompShg,MetalSd,MetalSd,Stone,680,Excellent,Typical,PConc,Excellent,Typical,Gd,GLQ,3,Unf,0,278,2535,GasA,Excellent,Y,SBrkr,2470,0,0,2470,2,0,1,1,1,1,Excellent,7,Typ,2,Good,Attchd,Fin,3,789,Typical,Typical,Paved,154,65,0,0,216,144,Excellent,No_Fence,NaN,0,2,2008,WD,Normal,615000,-93.656958,42.058484,325826.183516
1637,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,52,51974,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,CulDSac,Gtl,Stone_Brook,PosN,Norm,OneFam,One_Story,Excellent,Average,2006,2007,Hip,CompShg,VinylSd,VinylSd,BrkFace,710,Excellent,Typical,PConc,Excellent,Typical,Av,GLQ,3,Unf,0,1559,2660,GasA,Excellent,Y,SBrkr,2338,0,0,2338,1,0,2,1,4,1,Good,8,Typ,2,Good,Attchd,Fin,3,1110,Good,Typical,Paved,0,135,0,0,322,0,No_Pool,No_Fence,NaN,0,6,2007,New,Partial,591587,-93.626366,42.061202,317156.791630


In [5]:
ames.assign(resid = model.resid).nsmallest(3, "resid")

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,Land_Slope,Neighborhood,Condition_1,Condition_2,Bldg_Type,House_Style,Overall_Qual,Overall_Cond,Year_Built,Year_Remod_Add,Roof_Style,Roof_Matl,Exterior_1st,Exterior_2nd,Mas_Vnr_Type,Mas_Vnr_Area,Exter_Qual,Exter_Cond,Foundation,Bsmt_Qual,Bsmt_Cond,Bsmt_Exposure,BsmtFin_Type_1,BsmtFin_SF_1,BsmtFin_Type_2,BsmtFin_SF_2,Bsmt_Unf_SF,Total_Bsmt_SF,Heating,Heating_QC,Central_Air,Electrical,First_Flr_SF,Second_Flr_SF,Low_Qual_Fin_SF,Gr_Liv_Area,Bsmt_Full_Bath,Bsmt_Half_Bath,Full_Bath,Half_Bath,Bedroom_AbvGr,Kitchen_AbvGr,Kitchen_Qual,TotRms_AbvGrd,Functional,Fireplaces,Fireplace_Qu,Garage_Type,Garage_Finish,Garage_Cars,Garage_Area,Garage_Qual,Garage_Cond,Paved_Drive,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,Three_season_porch,Screen_Porch,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Year_Sold,Sale_Type,Sale_Condition,Sale_Price,Longitude,Latitude,resid
1498,Two_Story_1946_and_Newer,Residential_Low_Density,313,63887,Pave,No_Alley_Access,Irregular,Bnk,AllPub,Corner,Gtl,Edwards,Feedr,Norm,OneFam,Two_Story,Very_Excellent,Average,2008,2008,Hip,ClyTile,Stucco,Stucco,Stone,796,Excellent,Typical,PConc,Excellent,Typical,Gd,GLQ,3,Unf,0,466,6110,GasA,Excellent,Y,SBrkr,4692,950,0,5642,2,0,2,1,3,1,Excellent,12,Typ,3,Good,Attchd,Fin,2,1418,Typical,Typical,Paved,214,292,0,0,0,480,Good,No_Fence,NaN,0,1,2008,New,Partial,160000,-93.674898,42.016804,-483467.187204
2180,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,128,39290,Pave,No_Alley_Access,Slightly_Irregular,Bnk,AllPub,Inside,Gtl,Edwards,Norm,Norm,OneFam,One_Story,Very_Excellent,Average,2008,2009,Hip,CompShg,CemntBd,CmentBd,Stone,1224,Excellent,Typical,PConc,Excellent,Typical,Gd,GLQ,3,Unf,0,1085,5095,GasA,Excellent,Y,SBrkr,5095,0,0,5095,1,1,2,1,2,1,Excellent,15,Typ,2,Good,Attchd,Fin,3,1154,Typical,Typical,Paved,546,484,0,0,0,0,No_Pool,No_Fence,Elev,17000,10,2007,New,Partial,183850,-93.676220,42.016453,-398520.568735
2181,Two_Story_1946_and_Newer,Residential_Low_Density,130,40094,Pave,No_Alley_Access,Slightly_Irregular,Bnk,AllPub,Inside,Gtl,Edwards,PosN,PosN,OneFam,Two_Story,Very_Excellent,Average,2007,2008,Hip,CompShg,CemntBd,CmentBd,Stone,762,Excellent,Typical,PConc,Excellent,Typical,Gd,GLQ,3,Unf,0,878,3138,GasA,Excellent,Y,SBrkr,3138,1538,0,4676,1,0,3,1,3,1,Excellent,11,Typ,1,Good,BuiltIn,Fin,3,884,Typical,Typical,Paved,208,406,0,0,0,0,No_Pool,No_Fence,NaN,0,10,2007,New,Partial,184750,-93.676241,42.016642,-350820.782376


* Residuals sum to 0 because of how OLS picks the line. If they summed to something nonzero, say +5k on average, the line would be sitting too low, and I could slide it up by that amount (same slope, new intercept) and shrink the squared error. OLS has already used up that free up/down move, so at the minimum the sum has to be exactly 0. Any line shifted away from this one has a nonzero residual sum and a larger squared error, so a zero sum is a consequence of minimizing squared error.
* Residuals are uncorrelated with Gr_Liv_Area (the predictor) because OLS also stops tilting the line only when no pattern is left to exploit. If bigger houses still tended to have positive residuals, a steeper slope would reduce the squared error. So whatever linear relationship with Gr_Liv_Area existed has been absorbed into the coefficient, and what is left in the residuals is unrelated to it. (They are *not* uncorrelated with Sale_Price: residual + fitted = Sale_Price.)
* Largest positive residuals (index 44, 1063, 1637, +317k to +334k): all one-story homes of about 2,300-2,500 sqft, Excellent / Very_Excellent quality, Partial sales, in upscale neighborhoods (Northridge_Heights, Stone_Brook). These are under-predicted: quality and location add value that sqft alone can't see.
* Largest negative residuals (index 1498, 2180, 2181, -351k to -483k): all in Edwards, Very_Excellent quality, Partial sales, but enormous (4,676-5,642 sqft, almost double the top group). The line predicts roughly $540k-$640k and they sold for $160k-$185k. These are over-predicted: the straight line extrapolates linearly into the far tail of size, and Partial sales may be unfinished or unusual transactions.
* What separates the two groups is not quality or sale condition (both are Excellent+ and Partial) but size and neighborhood. Gr_Liv_Area alone can't capture quality, location, or sale circumstances, so the model forces all of that into one variable, which shows up as large residuals at both ends.

## 2. Assessing the Model — RMSE and R²

Two summaries of "how good is this line":

- **RMSE** (root mean squared error) is the typical size of a miss, in the target's units. If three residuals are +30k, -40k, +50k, square them (900M, 1.6B, 2.5B), average (1.67B), take the root: about $40.8k. Squaring means a single $200k miss counts four times what a $100k miss does, so RMSE is more sensitive to big misses than a plain average absolute miss.
- **R²** is the share of the target's wiggle that the model accounts for: `1 - (sum of squared residuals) / (sum of squared gaps from the plain mean)`. Predicting everyone at the average price is the "know-nothing" benchmark, and R² measures how much better than that benchmark you did. R² = 0 means no better than the mean; R² = 1 means perfect.

(`statsmodels` reports a *residual standard error* that divides by n - p instead of n; for n = 2,930 it's nearly identical to RMSE.)

**Question 2a.** For the model from Question 1: compute RMSE and R² *from the residuals by hand* (don't read them off the summary), then confirm they match `.rsquared` and `np.sqrt(.mse_resid)`. Compare the RMSE to the standard deviation of `Sale_Price` itself: what is the model buying you, in dollars? Is a typical miss of that size good or bad? What would you need to know to decide?

*Tools for this question:*
- `res.rsquared`; `np.sqrt(res.mse_resid)` (divides by n-p, so it differs slightly from `np.sqrt(np.mean(res.resid**2))`, which divides by n).
- By hand: `sst = ((y - y.mean())**2).sum()`, `sse = (res.resid**2).sum()`, `r2 = 1 - sse/sst`.

In [6]:
# compute RSME by hand
RMSE = np.sqrt((((model.resid)**2).sum())/len(ames))
print(f"Root mean squared error:{RMSE}")

# compute R^2 by hand
sse = ((model.resid)**2).sum()
sst = ((ames["Sale_Price"] - ames["Sale_Price"].mean())**2).sum()
R_squared = 1 - (sse/sst)
print(f"R-squared: {R_squared}")

Root mean squared error:56504.876669063226
R-squared: 0.49953785669572837


In [7]:
# package computed
print(f"R-squared: {model.rsquared}")

print(f"Root mean squared error:{np.sqrt(model.mse_resid)}")
print(f"Std dev of sale price: {np.std(ames["Sale_Price"])}")

R-squared: 0.4995378566957285
Root mean squared error:56524.17148836182
Std dev of sale price: 79873.05865192253


* Standard deviation shows that models deviate 79.8k from the average house price. Model root mean squared error is about 56.5k. This means that the difference, on average, of making predictions based on the mean only is about 23.3k more in error in comparison to utilizing the model. To be honest, a miss of either size (even in the context of housing) is pretty bad. In order to decide whether the model is worth it we would need to examine external factors to see how much an incorrect prediction would cost the initiative. 

**Question 2b.** RMSE and R² can tell different stories. Fit the same model on `log(Sale_Price)` (use `np.log(Sale_Price)` in the formula). What happened to R²? Can you compare this model's RMSE to the one from 2a? Why or why not? (Don't convert yet — just think about what units the residuals are in.) Put your reasoning in a markdown cell.

*Tools for this question:*
- Transforms work inside the formula: `smf.ols("np.log(Sale_Price) ~ Gr_Liv_Area", data=df)`.

In [8]:
# part 1 fit the model
log_model = smf.ols("np.log(Sale_Price) ~ Gr_Liv_Area",data = ames).fit()
print(f"Root mean squared error:{np.sqrt(log_model.mse_resid)}")
log_model.summary()

Root mean squared error:0.29276878811276785


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     np.log(Sale_Price)   R-squared:                       0.484
Model:                            OLS   Adj. R-squared:                  0.484
Method:                 Least Squares   F-statistic:                     2749.
Date:                Thu, 15 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:05:14   Log-Likelihood:                -557.36
No. Observations:                2930   AIC:                             1119.
Df Residuals:                    2928   BIC:                             1131.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      11.1795      0.017    660.123      0.000      11.146      11.213
Gr_Liv_Area     0.0006   1.07e-05     52.430      0.000       0.001       0.001
==============================================================================
Omnibus:                      684.292   Durbin-Watson:                   1.044
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             3518.272
Skew:                          -1.016   Prob(JB):                         0.00
Kurtosis:                       7.969   Cond. No.                     4.96e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.96e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

* R squared on the log sale price model went from 0.499 to 0.484, a small drop, so Gr_Liv_Area explains a similar share of the variation either way. R-squared is unitless so it is easier to compare than RMSE, but it is still not a perfectly like-for-like comparison: the two models are explaining the variance of different targets (price vs. log price). I would read it as "about the same", not "the log model is worse".
* Our RMSE went from 56,524 to 0.293. This does not mean the model makes fewer errors. In the old model the RMSE was in dollars, in the new one it is in log-dollars, so the two numbers are not comparable.
* A log-scale RMSE does have a meaning of its own: a residual of about 0.29 in log units is roughly a 29% miss in relative terms, a percentage error rather than a dollar error. For housing that is often the more natural way to judge a model, since a $50k miss matters very differently on a $100k house than on a $500k one.

## 3. Multiple Linear Regression

Add more predictors and the model fits a *plane* (or hyperplane): `Sale_Price ≈ b0 + b1*Gr_Liv_Area + b2*Year_Built + ...`. The crucial change in meaning: each coefficient is now the effect of that variable **holding the others fixed**. Numeric picture: if the model says $90/sqft for living area and $700/year for build year, then two houses that are the same age but differ by 100 sqft are predicted to differ by $9,000 — and two houses with identical sqft built 10 years apart differ by $7,000. Each slope answers its own "all else equal" question.

**Question 3.** Fit `Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF`. (a) Compare the `Gr_Liv_Area` coefficient to the one in Question 1. Did it move? Which direction, and why would adding *Year_Built* and *Total_Bsmt_SF* change what "an extra square foot of living area" is worth? (b) Interpret the `Year_Built` coefficient in dollars in one sentence. (c) The intercept is a huge negative number. Why — and what does it say about whether the intercept is something to interpret? (d) Compute RMSE and R² for this model. By how much did RMSE improve over Question 2a — a lot, a little?

*Tools for this question:*
- Add predictors with `+`: `"Sale_Price ~ A + B + C"`. `res.summary().tables[1]` prints just the coefficient table.

In [9]:
# build MLR 
MLR_Model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF",data = ames).fit()
print(f"MLR Model Coefficients:\n {MLR_Model.params}\n")


# get the coefficient from the first model
print(f"Coefficients of model 1:\n {model.params}")

MLR Model Coefficients:
 Intercept       -1.393589e+06
Gr_Liv_Area      7.118436e+01
Year_Built       7.057294e+02
Garage_Area      6.645799e+01
Total_Bsmt_SF    4.278171e+01
dtype: float64

Coefficients of model 1:
 Intercept      13289.634365
Gr_Liv_Area      111.694001
dtype: float64


**MLR vs Simple LR OLS Result Interpretation:**
* Gr_Liv_Area's coefficient went from $111.69 to $71.18. The reason is confounding: bigger homes also tend to have bigger basements and garages (and are somewhat newer), and those raise price too. In the simple model, an extra square foot of living area "carried along" all of that, so the $111.69 was not the value of a square foot alone. In the multiple model, $71.18 is the price difference for an extra square foot among homes with the same build year, basement size, and garage size, which is the "all else equal" reading.
* Year_Built in plain English: among homes with the same living area, garage, and basement, a home built one year later sold for about $706 more on average.
* The intercept is hugely negative because it is the prediction when every predictor is 0, including Year_Built = 0, about 1,900 years outside the data (1872-2010). A home built around 1970 gets roughly $706 x 1,970 = $1.39M from the Year_Built term alone, and the intercept has to cancel that out to bring predictions back to realistic prices. It is just where the line crosses zero, not the price of any real house, so it is not something to interpret. Centering Year_Built (subtracting, say, 1970) would turn it into the price of a typical-age home with zero of everything else.

In [10]:
# MLR r2 and RMSE
print(f"MLR R squared: {MLR_Model.rsquared}")
print(f"MLR RMSE: {np.sqrt((((MLR_Model.resid)**2).sum())/len(ames))}\n")

# model 1 r2 and RMSE
print(f"MLR R squared: {model.rsquared}")
print(f"MLR RMSE: {np.sqrt((((model.resid)**2).sum())/len(ames))}")

MLR R squared: 0.731537058804274
MLR RMSE: 41384.95502071521

MLR R squared: 0.4995378566957285
MLR RMSE: 56504.876669063226


* R-squared improved a lot, from 0.499 to 0.732, so the new predictors explain much more of the variation in price. R-squared never goes down when you add a variable, so whether this is a real improvement is what Question 4 tests.
* RMSE also dropped by about 15k (from about 56.5k to 41.4k).

## 4. In-sample fit vs. out-of-sample fit; adjusted R²

R² can only go up when you add a variable — even a useless one — because the least-squares fit can always use a bit of noise to shave the training residuals. Here's the numeric picture: with 3 data points a line can't generally hit all three, but with 3 data points and 2 predictors plus an intercept you can hit them *exactly* (R² = 1), no matter what the predictors are. In general, as the number of columns approaches the number of rows, the model can memorize.

**Adjusted R²** charges a toll for each added predictor: `1 - (1 - R²) * (n - 1) / (n - p - 1)`. If a predictor doesn't improve fit by more than chance would, adjusted R² falls.

**Question 4.** Split the data into 70% train / 30% test (`train_test_split` with a fixed `random_state`). On the train set, start from `Sale_Price ~ Gr_Liv_Area + Year_Built` and add pure-noise columns (`np.random.normal`, new column each time) in batches: 0, 10, 30, 100, 300 columns. For each batch record **train R², train adjusted R², and test RMSE** (predict on the test set with `.predict`). Make a small table or plot.
(a) Describe in words what each of the three quantities does as the number of noise columns grows. (b) At what point, if any, does adjusted R² fail to warn you? (c) What's the single-sentence moral about reporting R² computed on the same data used for fitting?

*Tools for this question:*
- `train_test_split(df, test_size=0.3, random_state=0)` returns `(train, test)`; `res.predict(test_df)` returns predictions for new rows; `res.rsquared_adj`.
- Noise columns: `rng = np.random.default_rng(0); rng.normal(size=len(train))`. Build the formula from a list: `"Sale_Price ~ " + " + ".join(cols)`.

In [11]:
from sklearn.metrics import mean_squared_error, r2_score

train_data, test_data = train_test_split(ames, test_size=0.3, random_state=42)
rng = np.random.default_rng(seed=0)

# Ensure we have mutable copies for the loop
train_df = train_data.copy()
test_df = test_data.copy()

noise_counts = [0,10,30,100,300]

results_log = []

for count in noise_counts:
    base_cols = ["Gr_Liv_Area", "Year_Built"]
    noise_cols = []
    
    # 3. Add identical pure noise columns to BOTH train and test data
    for i in range(1, count + 1):
        col_name = f"noise_{i}"
        train_df[col_name] = rng.normal(0, 1, size=len(train_df))
        test_df[col_name] = rng.normal(0, 1, size=len(test_df))
        noise_cols.append(col_name)
        
    # 4. Fit using your string formula idea
    formula = f"Sale_Price ~ {' + '.join(base_cols + noise_cols)}"
    model = smf.ols(formula=formula, data=train_df).fit()
    
    # 5. Predict on training data and test data
    y_train_pred = model.predict(train_df)
    y_test_pred = model.predict(test_df)
    
    # 6. Extract all 4 metrics
    train_r2 = model.rsquared
    adj_r2 = model.rsquared_adj
    pred_r2 = r2_score(test_df["Sale_Price"], y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(test_df["Sale_Price"], y_test_pred))
    
    # Keep track of metrics for graphing
    results_log.append({
        "noise_count": count, 
        "train_r2": train_r2, 
        "adj_r2": adj_r2, 
        "pred_r2": pred_r2, 
        "test_rmse": test_rmse
    })
    
    print(f"\nModel: 2 Predictors + {count} Noise Columns")
    print(f"-> Train R-squared:     {train_r2:.4f}  (Tricked by noise)")
    print(f"-> Adjusted R-squared:  {adj_r2:.4f}  (Penalizes noise elements)")
    print(f"-> Predicted R-squared: {pred_r2:.4f}  (Crashes as model overfits)")
    print(f"-> Test RMSE:          ${test_rmse:,.2f} (Swells up due to error)")


Model: 2 Predictors + 0 Noise Columns
-> Train R-squared:     0.6558  (Tricked by noise)
-> Adjusted R-squared:  0.6555  (Penalizes noise elements)
-> Predicted R-squared: 0.6626  (Crashes as model overfits)
-> Test RMSE:          $48,697.50 (Swells up due to error)

Model: 2 Predictors + 10 Noise Columns
-> Train R-squared:     0.6564  (Tricked by noise)
-> Adjusted R-squared:  0.6543  (Penalizes noise elements)
-> Predicted R-squared: 0.6611  (Crashes as model overfits)
-> Test RMSE:          $48,805.98 (Swells up due to error)

Model: 2 Predictors + 30 Noise Columns
-> Train R-squared:     0.6609  (Tricked by noise)
-> Adjusted R-squared:  0.6555  (Penalizes noise elements)
-> Predicted R-squared: 0.6580  (Crashes as model overfits)
-> Test RMSE:          $49,031.95 (Swells up due to error)


C:\Users\Bao\AppData\Local\Temp\ipykernel_27188\3909256223.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col_name] = rng.normal(0, 1, size=len(train_df))
C:\Users\Bao\AppData\Local\Temp\ipykernel_27188\3909256223.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col_name] = rng.normal(0, 1, size=len(test_df))



Model: 2 Predictors + 100 Noise Columns
-> Train R-squared:     0.6687  (Tricked by noise)
-> Adjusted R-squared:  0.6514  (Penalizes noise elements)
-> Predicted R-squared: 0.6443  (Crashes as model overfits)
-> Test RMSE:          $50,000.71 (Swells up due to error)


C:\Users\Bao\AppData\Local\Temp\ipykernel_27188\3909256223.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[col_name] = rng.normal(0, 1, size=len(train_df))
C:\Users\Bao\AppData\Local\Temp\ipykernel_27188\3909256223.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[col_name] = rng.normal(0, 1, size=len(test_df))



Model: 2 Predictors + 300 Noise Columns
-> Train R-squared:     0.7097  (Tricked by noise)
-> Adjusted R-squared:  0.6595  (Penalizes noise elements)
-> Predicted R-squared: 0.6087  (Crashes as model overfits)
-> Test RMSE:          $52,444.54 (Swells up due to error)


**Interpretation**
* As we increase the number of random noise features, r-square increase. This gives the impression that the model is "capturing" more variance as we increase the number for predictors even though they are random noise. This just shows that its fitting to the noise. 
* Adjusted r-squared penalizes for increase predictors and we can see that it stays stagnant which indicates that features we are adding are not helping the model explain more of the variance in the dependent variable.
* For RSME, the error on unseen data gets worse as noise is added, even while train R² improves. That gap is the overfitting.
* I would say that the adjusted R-squared did in fact try to warn me. As I add more noisy features the metric indicates little to no change in the models ability to explain the variance caused by the independent features. However for batch of 300 adjusted r-squared went up which would have failed to warn me. 
* Single sentence: be careful reporting trained r-squared as it might give you the impression that the features you are adding are helping the model even if they are not. It tells you how well it memorized patterns from the training set so the test set metrics are the results you should measure.

## 5. Model Selection and Stepwise Regression

With 15 candidate predictors there are 32,768 possible subsets, so people search greedily: **forward selection** adds whichever variable most improves the criterion, one at a time; **backward elimination** starts with everything and drops the least useful; **stepwise** can do both. The criterion is usually **AIC** (Akaike information criterion): roughly "fit quality minus a penalty for each extra parameter" — lower is better. It's like adjusted R² but derived from likelihood and generally a stronger penalty.

Python has no built-in `stepAIC`. Two tools to know:
- Write the loop yourself: fit `smf.ols` for every remaining candidate, keep the lowest `.aic`, repeat until nothing improves it.
- `sklearn.feature_selection.SequentialFeatureSelector` does the same greedy search but ranks candidates by **cross-validated** score rather than AIC.

**Question 5.** Candidates: `Gr_Liv_Area, Year_Built, Garage_Area, Total_Bsmt_SF, Lot_Area, Full_Bath, Fireplaces, Bedroom_AbvGr, TotRms_AbvGrd, Wood_Deck_SF, Open_Porch_SF, Year_Remod_Add, Garage_Cars, First_Flr_SF, Mas_Vnr_Area`.
(a) Write a forward-selection loop that adds the lowest-AIC variable at each step until AIC stops improving. Record the order of entry. (b) Run `SequentialFeatureSelector` (forward, `scoring='neg_root_mean_squared_error'`, `cv=5`) for the same number of features and compare the chosen sets — same or different? (c) Look at the variables that entered. Some candidates carry nearly the same information (e.g., `Garage_Cars` / `Garage_Area`, `Gr_Liv_Area` / `TotRms_AbvGrd`). How does the order of entry reflect that? (d) What does the selected model's p-value table actually mean, given that the same data chose the variables and then tested them?

*Tools for this question:*
- `res.aic` (lower is better). Loop idea: fit one model per remaining candidate, keep the lowest `.aic`.
- `SequentialFeatureSelector(LinearRegression(), n_features_to_select=k, direction="forward", scoring="neg_root_mean_squared_error", cv=5).fit(X, y)`; then `.get_support()` is a boolean mask over the columns of `X`. Imports: `sklearn.feature_selection`, `sklearn.linear_model`.

In [12]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

# build the loop
original_candidates = ["Gr_Liv_Area", 
              "Year_Built", 
              "Garage_Area", 
              "Total_Bsmt_SF", 
              "Lot_Area", 
              "Full_Bath", 
              "Fireplaces", 
              "Bedroom_AbvGr", 
              "TotRms_AbvGrd", 
              "Wood_Deck_SF", 
              "Open_Porch_SF", 
              "Year_Remod_Add", 
              "Garage_Cars", 
              "First_Flr_SF", 
              "Mas_Vnr_Area"]

train_df, test_df = train_test_split(ames, test_size=0.2, random_state=42)
train_df = train_df.copy()

# Use a copy for the AIC loop to preserve the original list
candidates = original_candidates.copy()

# --- Run Manual AIC Forward Selection ---
selected_features_aic = []
baseline_model = smf.ols(formula="Sale_Price ~ 1", data=train_df).fit()
current_best_aic = baseline_model.aic

while len(candidates) > 0:
    aic_tracker = {col: float("inf") for col in candidates}
    for col in candidates:
        formula = f"Sale_Price ~ {' + '.join(selected_features_aic)} + {col}" if selected_features_aic else f"Sale_Price ~ {col}"
        try:
            model = smf.ols(formula=formula, data=train_df).fit()
            aic_tracker[col] = model.aic
        except Exception:
            pass
    best_candidate = min(aic_tracker, key=aic_tracker.get)
    if aic_tracker[best_candidate] < current_best_aic:
        selected_features_aic.append(best_candidate)
        candidates.remove(best_candidate)
        current_best_aic = aic_tracker[best_candidate]
    else:
        break

# --- Run Scikit-Learn SequentialFeatureSelector ---
X_train = train_df[original_candidates]
y_train = train_df["Sale_Price"]

sfs = SequentialFeatureSelector(
    estimator=LinearRegression(),
    n_features_to_select=len(selected_features_aic),
    direction="forward",
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
sfs.fit(X_train, y_train)
selected_features_sfs = list(X_train.columns[sfs.get_support()])

# --- Print the Comparison Result ---
print("\n==================================================")
print(f"AIC Selected Features ({len(selected_features_aic)}): {selected_features_aic}")
print(f"SFS Selected Features ({len(selected_features_sfs)}): {selected_features_sfs}")
print("==================================================")



AIC Selected Features (11): ['Gr_Liv_Area', 'Year_Built', 'Total_Bsmt_SF', 'Garage_Cars', 'Bedroom_AbvGr', 'Year_Remod_Add', 'Fireplaces', 'Mas_Vnr_Area', 'Lot_Area', 'Wood_Deck_SF', 'Garage_Area']
SFS Selected Features (11): ['Gr_Liv_Area', 'Year_Built', 'Garage_Area', 'Total_Bsmt_SF', 'Lot_Area', 'Fireplaces', 'Bedroom_AbvGr', 'Wood_Deck_SF', 'Year_Remod_Add', 'Garage_Cars', 'Mas_Vnr_Area']


* **(b)** AIC forward selection and `SequentialFeatureSelector` picked the **same 11 variables**; both left out `Full_Bath`, `TotRms_AbvGrd`, `Open_Porch_SF`, and `First_Flr_SF`. SFS was told to pick exactly 11, so this compares the chosen sets, not where each method would stop on its own.
* **(c)** Order of entry decides which of two near-duplicates gets credit. `Garage_Cars` entered 4th, and `Garage_Area` only entered **last (11th)**, because once `Garage_Cars` was in, `Garage_Area` could add only the small bit of information `Garage_Cars` doesn't carry. `TotRms_AbvGrd` never entered, because `Gr_Liv_Area` went in first and already covered almost everything it would add. So a duplicate gets pushed down the order or left out, not locked out entirely, and which member of a pair wins is close to arbitrary (it can change from sample to sample).
* **(d)** The p-values of the selected model are **too optimistic**. A p-value assumes the model was chosen before looking at the data, but here each variable was picked *because* it looked strong on this data and then tested on that same data. The variables that look strongest by chance are the ones that get selected, so their p-values come out too small. The fix is to evaluate on data the selection never saw (a holdout split, or select on one half and test on the other).

## 6. Weighted Regression

Ordinary least squares counts every house's squared miss equally. **Weighted least squares** multiplies each squared miss by a weight first: `minimize sum of w_i * (residual_i)^2`. A house with weight 3 pulls on the line as if it appeared three times. Two common reasons to do it: observations differ in *reliability* (noisier ones should get smaller weights), or you simply care more about some observations (the book weights more recent sales more heavily, since last year's market says more about next year's price than ten-year-old sales).

Numeric picture: two houses at the same sqft, one a 2006 sale at $150k and one a 2010 sale at $180k. If the market moved up, OLS splits the difference and predicts $165k; with weights 1 for 2006 and 3 for 2010, the prediction becomes (150 + 3*180)/4 = $172.5k — pulled toward the recent price.

**Question 6.** Using `Year_Sold` (2006-2010), make a weight that increases with recency (e.g., `w = Year_Sold - 2005`). Fit `Sale_Price ~ Gr_Liv_Area + Year_Built` by OLS and by WLS (`smf.wls(..., weights=w)`). (a) Compare coefficients. Did the weights move the price-per-sqft estimate in the direction you'd expect given what you know about the market from 2006-2010? (Check by fitting OLS separately on 2006 only and 2010 only first, so you have a baseline.) (b) How do the standard errors and R² compare, and why can't you directly compare the R² values? (c) Name one situation where you'd choose WLS for *variance* reasons rather than *recency* reasons, and what you'd use as the weights.

*Tools for this question:*
- `smf.wls("y ~ x", data=df, weights=w).fit()` — `weights` is an array aligned with the rows; larger weight = that row counts more.
- `df.query("Year_Sold == 2006")` to fit on a single year.

In [30]:
# fit standard ols 2006 sales
model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built", data = ames[ames["Year_Sold"] == 2006]).fit()
print("2006 model:\n")
print(model.params)
print(model.rsquared)
print(model.bse)
print("\n")

# fit standard ols 2010 sales
model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built", data = ames[ames["Year_Sold"] == 2010]).fit()
print("2010 model:\n")
print(model.params)
print(model.rsquared)
print(model.bse)
print("\n")

# fit standard ols all years
model = smf.ols("Sale_Price ~ Gr_Liv_Area + Year_Built", data = ames).fit()
print("All years model:\n")
print(model.params)
print(model.rsquared)
print(model.bse)
print("\n")

2006 model:

Intercept     -1.860309e+06
Gr_Liv_Area    1.113785e+02
Year_Built     9.506503e+02
dtype: float64
0.7159561245698752
Intercept      119577.152189
Gr_Liv_Area         3.626091
Year_Built         61.365563
dtype: float64


2010 model:

Intercept     -2.198745e+06
Gr_Liv_Area    9.298906e+01
Year_Built     1.135303e+03
dtype: float64
0.6592281706745102
Intercept      158806.537265
Gr_Liv_Area         5.207956
Year_Built         81.376529
dtype: float64


All years model:

Intercept     -2.106459e+06
Gr_Liv_Area    9.596949e+01
Year_Built     1.087237e+03
dtype: float64
0.6590775951182664
Intercept      57338.740180
Gr_Liv_Area        1.757665
Year_Built        29.376908
dtype: float64




In [32]:
# define weighted column
ames["weight"] = ames["Year_Sold"] - 2005

# fit WLS
wls_model = smf.wls("Sale_Price ~ Gr_Liv_Area + Year_Built", data = ames, weights = ames["weight"]).fit()
print("WLS all years model:")
print(wls_model.params)
print(wls_model.rsquared)

WLS all years model:
Intercept     -2.164515e+06
Gr_Liv_Area    9.404749e+01
Year_Built     1.117972e+03
dtype: float64
0.6535890033580298


* **(a)** Price per sqft fell from **$111.38 in 2006** to **$92.99 in 2010** (OLS on each year alone), which is what I expected: 2006 sales were before the crash, 2010 sales were after it. Going from OLS on all years (**$95.97**) to WLS with recency weights (**$94.05**), the coefficient moved *toward* the 2010 value, which is the direction recency weights should push it. The shift is small because the weights only run from 1 to 5, and four of the five years carry real weight, so the blend barely leans toward 2010.
* **(b)** Standard errors are nearly identical (`Gr_Liv_Area`: 1.758 for OLS vs. 1.786 for WLS), so recency weights buy no precision; they only change which years the line favors. R² is 0.6591 (OLS) vs. 0.6536 (WLS), but these can't be compared directly because WLS's R² is computed from *weighted* sums of squares, which is a different scale. R² also isn't comparable across different samples (2006 vs. 2010) because it depends on how spread out prices are in that sample, not just on how well the model fits.
* **(c)** I'd choose WLS for **variance** reasons when some observations are noisier than others (heteroskedasticity). In Ames the residual SD grows with house size: about $24k for the smallest third, $33k for the middle, $69k for the largest. OLS gives all these houses equal say even though the large ones are far noisier. The fix is to weight each observation by the **inverse of its error variance**, e.g. `1 / Gr_Liv_Area**2` if the error SD grows in proportion to size. The other common case is when each row is an *average* of n items (a neighborhood mean), where the weight is n. R² being similar isn't a reason for or against weights; WLS is chosen to change what's estimated (recency) or to make estimates efficient and the standard errors correct (variance), not to raise R².

---
# Exam — check your intuition

7 questions. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, a few bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Reading a simple regression
*Role lean: DS*

A teammate fits `Sale_Price ~ Garage_Area` on all 2,930 Ames homes (`Garage_Area` in square feet):

```text
                 coef  std err      t   P>|t|
Intercept    68470.35  2737.27  25.01  <0.001
Garage_Area    237.65     5.27  45.09  <0.001

n = 2930   R-squared = 0.410   Adj. R-squared = 0.410   Residual std error = 61,384
Sale_Price std dev = 79,887
```

(a) In one sentence, what does the Garage_Area coefficient say, in dollars? (b) What does the intercept mean here, and would you put it in a slide? (c) The residual standard error is about 61,384 while the raw standard deviation of Sale_Price is about 79,887. What does comparing those two numbers tell you, and how does it connect to R-squared = 0.41? (d) A stakeholder says "R-squared of 0.41 means the model is wrong 59% of the time." What is wrong with that sentence?

**Your answer (Q1):**

* A: Garage_Area's coefficient states that, assuming all other variables are held constant, on average homes with 1 more foot of garage area are $237.65 more expensive.
* B: I would not put the intercept in a slide. It essentially states that a home with 0 would still have a sales price of $68.67k which would not be insightful enough to share.
* C: Comparing the numbers could imply if the choice was to use average sales price vs. this model to make predictions, the amount that our models "gaps" deviate from the mean is less than the actual standard deviation. This connects to r-squared becuase this metric desribes how well our model explains the variability of the sales price based on garage space which would be related to the gaps/residauls.
* D: This is incorrect. R-squared simply is our models ability to explain the variability in sales price with garage area as our only independent variable. Obviously, we would want a higher r-squared but to tie it directly to error is not right.

**Review (Q1):**

**Overall: right ideas. Two wording slips (a, b) and two answers to tighten (c, d).**

* **(a) Mostly right.** Drop "all other variables held constant": this model has no other variables. Say: *homes with 1 more sq ft of garage sell for $237.65 more on average.* That's an association, not what adding a garage would do.
* **(b) Right call, wrong reason.** Not slide-worthy, agreed. But the intercept isn't meaningless: 0 sq ft of garage is real in this data (158 homes have none), so $68,470 is the model's price for a **no-garage home**. It's a weak slide because it also absorbs everything else that differs for those homes. (Typo: $68.47k, not $68.67k.)
* **(c) Idea right, numbers missing.** 79,887 is your typical miss if you guess the mean for every house; 61,384 is your typical miss with the model, about **23% smaller**. R² turns that into variance: 1 − (61,384 / 79,887)² = 1 − 0.59 = **0.41**.
* **(d) Right verdict, wrong reason.** R² *is* tied to error (it's 1 − error² / spread²). The flaw is that it's a **share of variance, not a hit rate**: the model isn't "wrong" or "right" on a dollar amount, it's off by some amount every time. 59% is the share of price variation left unexplained.

### Exam Q2 — Units and rescaling
*Role lean: DS · interview*

Using the Garage_Area model above, predict what happens to each of the following if you re-fit after converting `Garage_Area` from square feet to **square meters** (1 sq m ≈ 10.76 sq ft): the slope, the intercept, R-squared, the t-statistic on the slope, and the predictions for any given house. Then: what if you instead rescaled `Sale_Price` to be in thousands of dollars? One line of reasoning per quantity is enough — if you know *why* each one moves or doesn't, you don't need to run anything.

**Your answer (Q2):**

* I would imagine that the slope increases. So if its about $237 per sqaure foot, and theres about 10-11 sq ft per sq meter then it would go up by a factor of 10-11x.
* I would imagine the intercept and r-square stays the same.
* If t-stat is just coefficient/SE of coefficient wouldn't that stay the same as well.
* If sales price went from 441,000 to 441 (in thousands) then i would imagine the coefficients of intercept and garage space decrease in size.

**Review (Q2):**

**Overall: right on every number you committed to. Two things to add: predictions, and how much the thousands rescale shrinks things.**

* **Slope (sq ft to sq m): Right.** A sq m is bigger, so one more *unit* is worth 10.76 × $237.65 ≈ **$2,557**.
* **Intercept, R², t-stat: Right.** Garage = 0 is 0 in any unit, and the slope and its SE are multiplied by the same 10.76, so their ratio is untouched.
* **Predictions: Missing.** They're **unchanged**: the model is the same line with relabeled units. *Rescaling a predictor changes its coefficient, not the model.*
* **Price in thousands: Mostly right.** Direction is right; be exact: slope and intercept both divide by **1,000** (about 0.238 and 68.47), and the standard errors divide by 1,000 too. R² and the t-stat are still unchanged, and predictions are the same prices just written in thousands.

### Exam Q3 — Extrapolation
*Role lean: DS · ML*

The same Garage_Area model predicts **$68,470** for a house with 0 sq ft of garage and **$781,411** for a house with 3,000 sq ft of garage. In the data, Garage_Area ranges from 0 to 1,488. Which of these two predictions would you trust more, and why? What specifically can go wrong when you predict outside the range of the data, even when R-squared and the residual plots inside the range look fine? How would you guard against it in a deployed model?

**Your answer (Q3):**

* I would say that the prediction using 0 sq ft is more trustworthy becuase even though our model is extremely simple and wouldn't perform amazing anyways, at least you're predicting using values it has seen before. 3000 is an extreme not in the range of data the model was fitted on.
* Predicting outside the range of data gives us a untrustworhty "far-reach" model attempt at extrapolating on a value it has never seen.
* To guard on extreme extrapolation, I would either limit the kind of predictions we are making or incorporate more features (so the model doesn't restrict itself to one number to predict off of).

**Review (Q3):**

**Overall: right choice and the right instinct. "What goes wrong" needs to be specific, and one guard doesn't work.**

* **Which to trust: Right.** $68,470 is at the edge of the data (158 homes have no garage); 3,000 sq ft is double the largest garage seen (1,488).
* **What goes wrong: Too vague.** Name the mechanism: the line *assumes the straight-line trend continues*, and R² and residual plots only vouch for the range you have. Real prices likely flatten, and $781,411 is already above every sale in the data (max $755,000). *Predicting outside the range is trusting an assumption the data can't check.*
* **Guard 1, limit predictions: Right.** Make it concrete: check inputs against the training range and **flag or refuse** anything outside it.
* **Guard 2, add more features: Off.** It doesn't remove extrapolation; it adds more ways to do it. A house can be inside every single feature's range and still be an unusual *combination* the model never saw. Better additions: monitor inputs for drift, and show prediction intervals (they widen away from the data).

### Exam Q4 — R-squared always goes up
*Role lean: DS · ML · interview*

To see what happens when you pile on useless features, a model is fit on a random **200-row** training sample using `Lot_Area`, `Year_Built`, and `k` columns of pure random noise (each generated independently of Sale_Price). Then it's scored on the other 2,730 homes it never saw:

```text
 noise_cols  R2_train  adjR2_train  RMSE_on_2730_unseen_rows
          0     0.380        0.374                     64303
          5     0.399        0.377                     65422
         20     0.461        0.394                     68275
         50     0.517        0.346                     72758
        100     0.759        0.505                     89242
        150     0.875        0.473                    133491
```

(a) Why does training R-squared climb to 0.88 with 150 columns of pure noise? (b) Adjusted R-squared is supposed to punish useless features. Does it fully protect you here? Look at what it says at k=100 versus the unseen-data RMSE at k=100 and say what you conclude about adjusted R-squared as a safeguard. (c) Interview version: "Your model has train R-squared 0.88 and performs terribly in production. Give me two explanations and tell me how you'd tell them apart." 

**Your answer (Q4):**

* unlike r-squared adjusted, regular r-squared does not penalize for the addition for more features. And so as you add more pure noise, it'll just find a way to fit better to that noise.
* while adjusted r-squared does a better job at remaining constant it does not protect you here. Someone looking at the jump from .346 to .505 would say that the model is improving but in reality its still just fitting to noise.
* R-squared is a metric computed from the data that it was trained on. So the first one that jumps to me is that fact that our train test split was extremely unbalanced. The model was trained on 200 rows, the r-squared was computed from its fit on those 200 rows, and it most likely performs terribly becuase the 2,730 new data points is something that the model was never trained on (in terms of training on values remotely similar). On the otherhand, if train test split was better and the model still performed poorly in production I would think about what kind of the data we are using in live predictions vs. what we're training on (i know this might sound similar to what I said before). To differentiate it, looking at test r-squared may be helpful.

**Review (Q4):**

**Overall: (a) and (b) are right. (c) names the right second cause but mislabels the first.**

* **(a) Right.** R² can't go down when you add a column, so noise gets soaked up as "signal".
* **(b) Right.** Sharpen it with the numbers: k=100 has the **highest adjusted R² in the table (0.505)** but an unseen RMSE about **39% worse** than the 2-feature model (89,242 vs. 64,303). Adjusted R² is still a *training-data* number with a fixed penalty, so it can't see out-of-sample failure.
* **(c) Mostly right.** Your second cause (live data differs from training data) is right: **distribution shift**. The first isn't an "unbalanced split"; a large test set is fine. The real cause is **overfitting**: 200 rows, ~150 columns, nearly one parameter per row, so the model memorized noise. *Tell them apart with a held-out set from the same period as training: if it's also bad, it's overfitting; if it's fine but production is bad, it's shift (or leakage).*

### Exam Q5 — Stepwise selection is unstable
*Role lean: DS · ML · interview*

Forward selection (add the variable that most lowers AIC, one at a time) is run twice on 15 candidate numeric features — once on one random 400-row sample of the data and once on a *different*, non-overlapping 400-row sample from the same population. The order variables were chosen in:

```text
step   half A (rows 0-399)    half B (rows 400-799)
1      Total_Bsmt_SF          Gr_Liv_Area
2      Gr_Liv_Area            Year_Built
3      Year_Built             Garage_Area
4      Bedroom_AbvGr          Total_Bsmt_SF
5      Kitchen_AbvGr          Screen_Porch
6      Bsmt_Unf_SF            Year_Remod_Add
```

(a) What does this tell you about trusting the "final model" that stepwise selection hands back? (b) Someone says "the variables stepwise picked are significant (p < 0.05) so they're the true drivers of price." What is wrong with that claim — what assumption behind the p-value did the selection process break? (c) Name two approaches you would prefer for deciding which features to keep (hint: one is for prediction, one is for explanation) and say when you'd reach for each.

**Your answer (Q5):**

* A: You should not consider the final model you receive the be the absolute best. Sometimes variables like which rows were used to compute feature AIC or even the order at which the features are introduced themselves (like the case where we had multiple features representing the same thing) can affect the final model.
* B. p-values in this sense are optimistic. P-values are computed on the data it has seen. To be more realistic, it should be computed on a holdout set.
* C. To be honest, for prediction sake, I believe incorporating all the features and then performing regularization might be of great help. For instance, L1 (lasso) allows for coefficients to shrink to 0 effectively performing feature selection. In terms of choosing which features to keep for explanability (im assuming in the context of casual inference), I would stick to mainly thinking about your main X and Y and then controlling for confounders. I believe interaction terms are used when moderators are present as well.

**Review (Q5):**

**Overall: strong. Tie (a) to the table and name the broken assumption in (b).**

* **(a) Right.** Point at the table: the two samples agree only on the top picks (`Gr_Liv_Area`, `Year_Built`), then **diverge completely** from step 4 (`Bedroom_AbvGr`, `Kitchen_AbvGr`, `Bsmt_Unf_SF` vs. `Total_Bsmt_SF`, `Screen_Porch`, `Year_Remod_Add`). The later picks are sample-specific noise.
* **(b) Mostly right.** Name the assumption: a p-value assumes the variables were chosen **before** looking at the data. Selection picked them *because* they looked significant here, so the p-values come out too small. Also add: even a real p-value says "associated with price", not "true driver".
* **(c) Right.** Lasso/ridge with the penalty chosen by cross-validation for prediction; for explanation, pick controls from **domain knowledge** (confounders, not mediators), not from the data. Interactions were a fine tangent but are about effect differences, not selection.

### Exam Q6 — When does weighting help?
*Role lean: DS · Causal*

For the Garage_Area model, the spread of the residuals (how far actual prices typically miss the fitted line) differs a lot by garage size:

```text
               n  mean_garage_area  resid_sd
tercile                                     
low garage   977             247.0   37578.0
mid garage   976             473.0   46062.0
high garage  977             698.0   87874.0
```

(a) A colleague suggests weighted least squares with weights proportional to 1 / (residual variance of the group). Which group gets the larger weight, and what's the intuition in terms of "how much should one observation's miss be allowed to pull the line"? (b) Would you expect the **coefficients** to change a lot, a little, or not at all? What would you expect to change more reliably? (c) Separately: in the book, weighted regression is also used to *down-weight old observations*. Give a situation in a real analysis (not necessarily housing) where you'd choose weights for that reason rather than for unequal variance.

**Your answer (Q6):**

* A: I would assume that since the high garage tercile has the higher residual_sd it would have the highest weight (this is assuming my understanding of weight is correct in the sense that weight penalizes/emphasizes the error based on the factor it's set to).
* B: From what I remember in relation to the 2006-2010 housing analysis, weighting did not change the coefficients a drastic amount. I would imagine that while coefficients might remain within a reasonable range of similarity, r-squared and RMSE would improve.
* C: Down weighting old obserations is super common in marketing and demand forecasting. For example, a propensity model that predicts consumers likelihood to purchase might consider their purchase history with more weight towards their latest trends (or in this case we would down weight their habits we tracked 4 years ago). In demand forecasting, older observations may be downweighted since those observations could be only representative of the brand/company at a older period of time when operations/population is not as big as whats current.

**Review (Q6):**

**Overall: (c) is right. (a) is backwards and (b) has one wrong claim.**

* **(a) Off, reversed.** Weight = 1 / variance, so the **noisiest group (high garage) gets the smallest weight**, and the low-garage group gets about **5.5×** as much. Your "weight multiplies the miss" idea is right; you applied it the wrong way round. *A big miss is normal where prices are noisy, so it's less informative and should pull the line less.*
* **(b) Half right.** Coefficients change little: right, but because OLS is still unbiased here, not because of the 2006–2010 analysis. What changes reliably is the **standard errors**: WLS gives more precise estimates and trustworthy intervals. "R² and RMSE would improve" is wrong: OLS already minimizes the unweighted error, so RMSE can't get better in-sample, and the two R² values aren't comparable.
* **(c) Right.** The relationship drifts over time, so recent data represents the present better.

### Exam Q7 — Correlation, causation, and the coefficient
*Role lean: Causal · DS*

A product manager reads the Garage_Area model and says: "Great — every extra square foot of garage adds about $238 to a home's value, so if we convince owners to build 200 sq ft garage additions, we'll add about $47,000 each." Without running any code: (a) name at least two reasons the regression coefficient might not be the effect of *building* a garage addition, (b) say which Ames columns you'd want in the model to start addressing your reasons, and (c) say what kind of data or design (beyond a bigger regression) would let you answer the PM's actual question.

In [34]:
ames.columns

Index(['MS_SubClass', 'MS_Zoning', 'Lot_Frontage', 'Lot_Area', 'Street',
       'Alley', 'Lot_Shape', 'Land_Contour', 'Utilities', 'Lot_Config',
       'Land_Slope', 'Neighborhood', 'Condition_1', 'Condition_2', 'Bldg_Type',
       'House_Style', 'Overall_Qual', 'Overall_Cond', 'Year_Built',
       'Year_Remod_Add', 'Roof_Style', 'Roof_Matl', 'Exterior_1st',
       'Exterior_2nd', 'Mas_Vnr_Type', 'Mas_Vnr_Area', 'Exter_Qual',
       'Exter_Cond', 'Foundation', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure',
       'BsmtFin_Type_1', 'BsmtFin_SF_1', 'BsmtFin_Type_2', 'BsmtFin_SF_2',
       'Bsmt_Unf_SF', 'Total_Bsmt_SF', 'Heating', 'Heating_QC', 'Central_Air',
       'Electrical', 'First_Flr_SF', 'Second_Flr_SF', 'Low_Qual_Fin_SF',
       'Gr_Liv_Area', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath',
       'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'Kitchen_Qual',
       'TotRms_AbvGrd', 'Functional', 'Fireplaces', 'Fireplace_Qu',
       'Garage_Type', 'Garage_Finish', 'Garage_Cars', 'G

**Your answer (Q7):**

* A: Two reasons I can think of start with the idea of confounding variables. The coefficient you see may not be reliable if garage_area is the only feature in that model. Adding features may/most likely would shrink the effect of garage_area. Second reason, the coefficient simply says homes with 1 more sq ft of garage on average see $238 of higher sales price. This does not imply the effect of building a garage from a house with no garage to start and I dont think plugging in X into the equation would give that net effect.
* B: I believe Lot_area, neighborhood, garage type, garage qual are just some of the many variables that could add additional information to the model that would help address the issue of trying to squeeze out all the information out of 1 feature along with having more granular view of what aspects of garages influence sales price.
* C: First thing that comes to mind would be something like an uplift model that demonstrates the impact of BUILDING garages. Also feature importance diagrams/shap waterfalls could better visualize the reasoning behind the model.

**Review (Q7):**

**Overall: (a) is right. (b) misses the biggest confounders. (c) reaches for the wrong tools.**

* **(a) Right.** Name them: **omitted-variable bias** (garage area stands in for house size, quality and neighborhood) and **between-house vs. within-house**. The coefficient compares *different* homes; the PM asks what happens to *one* home when you add a garage.
* **(b) Mostly right.** `Neighborhood` and `Lot_Area` are good, but add the strongest confounders: **`Gr_Liv_Area`, `Overall_Qual`, `Year_Built`**. `Garage_Type` / `Garage_Qual` describe the garage itself, so they refine the treatment rather than remove confounding.
* **(c) Right instinct, wrong tools.** You do need a *treatment effect* design. But SHAP explains the model, not the world, and an uplift model still needs treated and untreated groups. *Better: a randomized experiment (e.g., randomly offer a build subsidy), or before/after data on the same homes that built additions vs. similar homes that didn't (difference-in-differences).*